# Colab Full Control MCP Setup

This notebook starts the MCP server inside Colab, checks it locally, opens a Cloudflare Tunnel, and prints the Codex MCP config snippet.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

REPO_URL = 'https://github.com/CopyyQ/colab-full-control-mcp.git'
PROJECT_DIR = '/content/colab-full-control-mcp'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd /content/colab-full-control-mcp

Cloning into 'colab-full-control-mcp'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/colab-full-control-mcp'
/content


In [3]:
%pip install -r requirements.txt
%pip install -e .

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [6]:
import os
from getpass import getpass

os.environ['COLAB_MCP_TOKEN'] = getpass('COLAB_MCP_TOKEN: ')
os.environ['PERMISSION_PROFILE'] = 'DEVELOPER'
os.environ['ALLOWED_ROOTS'] = '/content,/content/drive/MyDrive'
os.environ['UNRESTRICTED_RUNTIME_MODE'] = 'false'

In [ ]:
import subprocess, sys

server_proc = subprocess.Popen(
    [sys.executable, 'scripts/start_server.py'],
    cwd='/content/colab-full-control-mcp',
)
print('server pid =', server_proc.pid)

In [ ]:
!python scripts/health_check.py

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb || apt-get -f install -y
!python scripts/start_tunnel.py --server-url http://127.0.0.1:8000

In [ ]:
import json
from pathlib import Path

state = json.loads(Path('/content/.colab_full_control_mcp/jobs/cloudflared_state.json').read_text())
public_url = state['url'] + '/mcp'
print('Public MCP URL:', public_url)
!python scripts/print_codex_config.py --url {public_url}

In [ ]:
from colab_full_control_mcp.tools import TOOL_REGISTRY

print('Registered tools:', sum(len(names) for names in TOOL_REGISTRY.values()))

In [ ]:
!python scripts/stop_tunnel.py
!python scripts/stop_server.py

In [7]:
!ls

drive  sample_data
